In [1]:
import json

from dotenv import load_dotenv
import os
from anthropic import Anthropic

load_dotenv()
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
# model = "claude-sonnet-4-5"
model = "claude-haiku-4-5"


def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {"model": model, "messages": messages, "max_tokens": 1000, "temperature": temperature}
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text

In [8]:
from statistics import mean
import re
import ast

def run_prompt(test_case):
    """merges prompt and test case input then return result"""
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}
    *respond onlyt with python, json or regex code
    *do not include any explanation or explanations or commentary
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    return chat(messages, stop_sequences=["```"])

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Solution Evaluation Criteria:
<solution_criteria>
{test_case["solution_criteria"]}
</solution_criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def run_test_case(test_case):
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "score": score,
        "test_case": test_case,
        "reasoning": reasoning,
    }

def run_eval(dataset):
    """loads dataset and runs test cases"""
    res = []
    for test_case in dataset:
        res.append(run_test_case(test_case))

    average_score = mean([r["score"] for r in res])
    print(f"Average score: {average_score}")
    return res

In [9]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)

Average score: 8.5


In [10]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport boto3\n\ndef list_s3_objects(bucket_name, prefix):\n    \"\"\"\n    Lists all object keys in an S3 bucket matching the given prefix.\n    \n    Args:\n        bucket_name (str): The name of the S3 bucket\n        prefix (str): The prefix to filter objects\n    \n    Returns:\n        list: A list of object keys matching the prefix\n    \"\"\"\n    s3_client = boto3.client('s3')\n    object_keys = []\n    \n    paginator = s3_client.get_paginator('list_objects_v2')\n    page_iterator = paginator.paginate(\n        Bucket=bucket_name,\n        Prefix=prefix\n    )\n    \n    try:\n        for page in page_iterator:\n            if 'Contents' in page:\n                for obj in page['Contents']:\n                    object_keys.append(obj['Key'])\n    except Exception as e:\n        print(f\"Error listing objects: {e}\")\n        return []\n    \n    return object_keys\n",
    "score": 8.5,
    "test_case": {
      "task": "Write a Python function that takes